# Lab 03 — Enterprise connectors and use-case assistants (solution)

> Reference solution: every TODO is filled in and every check passes.

**Level:** Intermediate → Advanced · **Duration:** ~90 minutes · **Covers:** H5 enterprise connector lab (45 min) · H6 policy-search and customer-support assistants from a shared pattern (45 min)

Lab guide: [`../lab-03-enterprise-connectors-assistants.md`](../lab-03-enterprise-connectors-assistants.md) — read each part there first; this notebook holds the code.

You connect the pipeline to SharePoint and Confluence (offline exports, the same shape the LlamaIndex readers return), keep it in sync, index it in Pinecone or Chroma with access metadata, and build two Daimler Truck assistants from one pattern. Your TODOs: the Confluence loader, the access filter and the assistant factory. You choose and justify the score floor.

In [ ]:
# (notebook) Setup: run this cell first. LLM_PROVIDER picks the model, EMBEDDING_PROVIDER the embeddings.
import json
import os
import sys
import time
import warnings
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))  # walks up from this folder to the repo-root .env
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")  # Chroma: no usage telemetry
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="IProgress not found")
pd.set_option("display.max_colwidth", 110)

# labs/module-03 holds data/ and rag_kit.py; the notebook runs from notebooks/ or solutions/.
LAB_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents) if p.name == "module-03" and (p / "data").is_dir())
DATA_DIR, OUTPUT_DIR = LAB_DIR / "data", LAB_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
sys.path[:0] = [str(LAB_DIR.parent / "setup"), str(LAB_DIR)]
import rag_kit as rk  # noqa: E402
from llm import (  # noqa: E402
    describe, embedding_slug, get_embedding_model_name, get_embedding_provider, get_provider,
    is_configured, partner_provider,
)
from transformers.utils import logging as hf_logging  # noqa: E402

hf_logging.disable_progress_bar()
PROVIDER, PARTNER = get_provider(), partner_provider()
if not is_configured(PROVIDER):
    raise SystemExit(
        f"❌ LLM_PROVIDER={PROVIDER} is not configured. Add its key and model to the repo-root .env "
        "(template: labs/setup/.env.example), restart the kernel and run this cell again."
    )
HAS_PARTNER = is_configured(PARTNER)


def check(name: str, fn) -> None:
    """Run a check function. A returned string means 'skipped'; a TODO is reported, not raised."""
    try:
        skipped = fn()
    except NotImplementedError as exc:
        print(f"⏳ {name}: TODO not implemented yet ({exc})")
        return
    print(f"⚠️  {name}: skipped ({skipped})" if skipped else f"✅ {name}")


print(describe())
print(f"Comparison provider: {PARTNER} {'✅' if HAS_PARTNER else '⚠️  not configured: comparison cells skip it'}")
print(f"Embeddings: {get_embedding_provider()} · {get_embedding_model_name()} (index suffix {embedding_slug()})")

from llama_index.core import QueryBundle, Settings
from llama_index.core.ingestion import DocstoreStrategy, IngestionPipeline
from llama_index.core.node_parser import MarkdownNodeParser
from llama_index.core.storage.docstore import SimpleDocumentStore

rk.setup_llama(PROVIDER)
BACKEND = rk.vector_backend()  # pinecone when PINECONE_API_KEY is set, otherwise chroma
print(f"Vector store: {BACKEND}" + ("" if BACKEND == "pinecone" else " (no PINECONE_API_KEY: the offline fallback)"))

## Part 1 — Connect SharePoint and Confluence (20 min)

A connector is a data pipeline: it reads as a service identity, keeps each document's **access
list**, and maps source-specific fields onto **one metadata schema**. Here the sources are offline
exports in `data/connectors/`; with tenant credentials the LlamaIndex Hub readers return the same shape.

In [ ]:
# (notebook) Live readers when the tenant is configured (instructor), otherwise the exports. No credentials in code.
import importlib.util

LIVE = {
    "SharePoint": ("llama_index.readers.microsoft_sharepoint", ["SHAREPOINT_CLIENT_ID", "SHAREPOINT_CLIENT_SECRET", "SHAREPOINT_TENANT_ID"]),
    "Confluence": ("llama_index.readers.confluence", ["CONFLUENCE_URL", "CONFLUENCE_API_TOKEN"]),
}
for source, (module, variables) in LIVE.items():
    live = importlib.util.find_spec(module.rsplit(".", 1)[0]) is not None and all(os.getenv(v) for v in variables)
    print(f"{source:10}: {'live reader available (instructor demo)' if live else 'offline export in data/connectors/ (no tenant configured)'}")

sharepoint = rk.load_sharepoint_export()
pd.DataFrame([{k: r["metadata"][k] for k in ("doc_id", "version", "status", "region", "audience", "allowed_groups")} for r in sharepoint])

In [ ]:
# (notebook) Your TODO: turn the Confluence export into records with the shared metadata schema.
def load_confluence_export(path=DATA_DIR / "connectors" / "confluence" / "pages.json") -> list[dict]:
    """One record per page: {"text": ..., "metadata": {...}}.

    - text: "# <page title>\n\n" + the storage-format body converted with rk.storage_to_text(...)
    - metadata: doc_id and title (split the page title at the first space: "ENG-SPEC-0417 ED-26 ..."),
      doc_type="engineering", region="ALL", audience="internal", owner_team="Engineering", language="en",
      updated (the date part of version.when), status="published", version (str of version.number),
      allowed_groups (restrictions.read.group, a list), source="confluence",
      url ("https://confluence.example/wiki" + _links.webui), product="ALL"
    """
    export = json.loads(Path(path).read_text(encoding="utf-8"))
    records = []
    for page in export["results"]:
        doc_id, _, title = page["title"].partition(" ")
        records.append({"text": f"# {page['title']}\n\n" + rk.storage_to_text(page["body"]["storage"]["value"]), "metadata": {
            "doc_id": doc_id, "title": title, "doc_type": "engineering", "region": "ALL", "audience": "internal",
            "owner_team": "Engineering", "language": "en", "updated": page["version"]["when"][:10],
            "status": "published", "version": str(page["version"]["number"]),
            "allowed_groups": page["restrictions"]["read"]["group"], "source": "confluence",
            "url": f"https://confluence.example/wiki{page['_links']['webui']}", "product": "ALL",
        }})
    return records

In [ ]:
# (notebook) Check: same schema as the corpus and SharePoint, access lists kept, no HTML left.
def check_part1():
    pages = load_confluence_export()
    assert [p["metadata"]["doc_id"] for p in pages] == ["ENG-SPEC-0417", "ENG-SPEC-0422", "ENG-SPEC-0440"]
    assert set(pages[0]["metadata"]) >= set(sharepoint[0]["metadata"]) - {"url"}
    assert pages[2]["metadata"]["allowed_groups"] == ["engineering"]
    assert "<td>" not in pages[0]["text"] and "| Cell temperature spread | > 8 K |" in pages[0]["text"]


check("Part 1 — Confluence connector", check_part1)
confluence = rk.todo_or_reference(load_confluence_export, rk.load_confluence_export)()
print(confluence[0]["text"][:400])

## Part 2 — Keep the index in sync: retired versions, edits and deletes (10 min)

The SharePoint library still holds the **retired** 2025 warranty policy. Indexing it would make the
assistant confidently quote the wrong driveline warranty.

In [ ]:
# (notebook) Drop retired versions at ingest, then run an ingestion pipeline with a docstore.
records = [r for r in rk.load_corpus() + sharepoint + confluence if r["metadata"].get("status", "published") == "published"]
retired = [r["metadata"]["doc_id"] + "@" + r["metadata"]["version"] for r in sharepoint if r["metadata"]["status"] != "published"]
print(f"{len(records)} published documents · excluded as retired: {retired}")

docs = rk.to_llama_documents(records)  # id_ = doc_id: the key for upserts and deletes
sync_store = rk.chroma_vector_store(f"lab03-sync-{embedding_slug()}")
pipeline = IngestionPipeline(
    transformations=[MarkdownNodeParser(), Settings.embed_model],  # chunk, then embed
    vector_store=sync_store, docstore=SimpleDocumentStore(), docstore_strategy=DocstoreStrategy.UPSERTS_AND_DELETE,
)
runs = {"first run": len(pipeline.run(documents=docs)), "unchanged rerun": len(pipeline.run(documents=docs))}
docs[0].set_content(docs[0].text + "\n\nUpdate 2026-10: also inspect the dosing line for kinks.")
runs["one document edited"] = len(pipeline.run(documents=docs))
runs["one document deleted"] = len(pipeline.run(documents=docs[1:]))
print({**runs, "vectors in the store now": sync_store._collection.count()})

## Part 3 — Index with access metadata, and prove the trimming (15 min)

The index runs on **Pinecone** when `PINECONE_API_KEY` is set, otherwise on **Chroma**: the
same LlamaIndex code, a different vector store. Access filters come from the authenticated
user, never from the model.

In [ ]:
# (notebook) Build the knowledge base on the chosen backend.
nodes = MarkdownNodeParser().get_nodes_from_documents(rk.to_llama_documents(records))
with rk.Timer() as t:
    index = rk.build_index(nodes, f"m03-{embedding_slug()}", backend=BACKEND, namespace="knowledge")
print(f"{len(nodes)} nodes indexed on {BACKEND} in {t.seconds}s")
if BACKEND == "pinecone" and os.getenv("PINECONE_REGION", "us-east-1").startswith("us-"):
    print("⚠️  Residency: this index is in a US region but holds EU documents. POL-DAT-004 §3 requires an EU region.")

In [ ]:
# (notebook) Your TODO: the metadata filter for one user. rk.access_filters is the reference.
from llama_index.core.vector_stores import FilterCondition, FilterOperator, MetadataFilter, MetadataFilters


def access_filters(user: rk.User, doc_types: list[str] | None = None) -> MetadataFilters:
    """AND of: region IN (user.region, "ALL"); audience IN user.audiences;
    OR over the user's groups plus "public" of acl_<group> == 1; and doc_type IN doc_types when given."""
    groups = [MetadataFilter(key=f"acl_{g}", value=1) for g in (*user.groups, "public")]
    filters = [
        MetadataFilter(key="region", value=[user.region, "ALL"], operator=FilterOperator.IN),
        MetadataFilter(key="audience", value=list(user.audiences), operator=FilterOperator.IN),
        MetadataFilters(filters=groups, condition=FilterCondition.OR),
    ]
    if doc_types:
        filters.append(MetadataFilter(key="doc_type", value=list(doc_types), operator=FilterOperator.IN))
    return MetadataFilters(filters=filters, condition=FilterCondition.AND)

In [ ]:
# (notebook) One question, three users: who sees which documents?
filters_for = rk.todo_or_reference(access_filters, rk.access_filters, rk.EU_DEALER)
QUESTION = "How long is the warranty on driveline components, and who approves goodwill above the dealer limit?"
trimmed = {}
for user in (rk.EU_DEALER, rk.US_WARRANTY, rk.FLEET_CUSTOMER):
    hits = index.as_retriever(similarity_top_k=6, filters=filters_for(user)).retrieve(QUESTION)
    trimmed[user.name] = hits
    print(f"{user.name:22} {sorted(set(rk.doc_ids(hits)))}")

In [ ]:
# (notebook) Check: every retrieved chunk is visible to its user; the internal goodwill policy never reaches the dealer.
def check_part3():
    access_filters(rk.EU_DEALER)  # your TODO must run, not just the reference
    for user in (rk.EU_DEALER, rk.US_WARRANTY, rk.FLEET_CUSTOMER):
        for hit in trimmed[user.name]:
            assert rk.visible_to(user, hit.node.metadata), (user.name, hit.node.metadata["doc_id"])
    assert "POL-WAR-001" in rk.doc_ids(trimmed["eu_dealer_advisor"]) and "POL-GDW-003" not in rk.doc_ids(trimmed["eu_dealer_advisor"])
    assert "POL-WAR-002" in rk.doc_ids(trimmed["us_warranty_analyst"]) and "POL-WAR-001" not in rk.doc_ids(trimmed["us_warranty_analyst"])


check("Part 3 — access trimming", check_part3)

## Part 4 — One pattern, two assistants (25 min)

The pattern is fixed: **filter → retrieve → re-rank → score floor → answer with citations**. A
profile decides what is searched and how the answer is written.

In [ ]:
# (notebook) Two profiles: an internal policy-search assistant and a fleet-customer support assistant.
POLICY_SEARCH = rk.RagProfile(
    "policy-search", ("policy", "contract"),
    "You answer Daimler Truck employees' and dealers' questions about warranty, goodwill, software and data "
    "policies and service contracts. Cite the document id and section, for example [POL-WAR-001 §3]. "
    "If two documents disagree, show both.",
)
CUSTOMER_SUPPORT = rk.RagProfile(
    "customer-support", ("faq",),
    "You help Daimler Truck fleet customers and drivers. Use plain, friendly language. Never mention internal "
    "teams, bulletins or systems that are not in the sources. Cite [KB-…] ids.",
)

In [ ]:
# (notebook) Your TODO: the shared pattern as a LlamaIndex query engine.
from llama_index.core import PromptTemplate
from llama_index.core.postprocessor import SimilarityPostprocessor


def make_assistant(profile: rk.RagProfile, user: rk.User, index, llm=None):
    """A query engine that: retrieves 2 * profile.k candidates with access_filters(user, profile.doc_types),
    re-ranks them to profile.k with rk.reranker(profile.k), drops nodes scoring below profile.floor
    (SimilarityPostprocessor), and answers with profile.instructions + rk.ANSWER_RULES in the prompt
    (a PromptTemplate with {context_str} and {query_str})."""
    prompt = PromptTemplate(
        f"{profile.instructions}\n{rk.ANSWER_RULES}\n\n<sources>\n{{context_str}}\n</sources>\n\nQuestion: {{query_str}}\nAnswer: "
    )
    return index.as_query_engine(
        llm=llm,
        similarity_top_k=2 * profile.k,
        filters=filters_for(user, list(profile.doc_types)),
        node_postprocessors=[rk.reranker(profile.k), SimilarityPostprocessor(similarity_cutoff=profile.floor)],
        text_qa_template=prompt,
    )

In [ ]:
# (notebook) Real questions for each assistant, including ones they must not answer.
CASES = {
    "policy-search": (POLICY_SEARCH, rk.EU_AFTERSALES, [
        ("How long is the HV battery warranty on an ED-26, and what capacity is guaranteed?", True),
        ("A customer asks for €4,000 goodwill on a gearbox 8 months after warranty. Who approves it?", True),
        ("What uptime does contract CTR-2026-118 guarantee, and what are the service credits?", True),
        ("What is the warranty on aftermarket trailers?", False),
    ]),
    "customer-support": (CUSTOMER_SUPPORT, rk.FLEET_CUSTOMER, [
        ("How long does a DC charge from 20 to 80 percent take?", True),
        ("The AdBlue lamp is flashing. Can I keep driving?", True),
        ("My truck's software update is stuck. What should I do?", True),
        ("Can I get a discount on my next truck?", False),
    ]),
}
build = rk.todo_or_reference(make_assistant, rk.make_assistant, POLICY_SEARCH, rk.EU_AFTERSALES, index)


def run_cases(provider: str) -> pd.DataFrame:
    llm = rk.llama_llm(provider)
    rows = []
    for name, (profile, user, questions) in CASES.items():
        assistant = build(profile, user, index, llm)
        for question, answerable in questions:
            with rk.Timer() as t:
                response = assistant.query(question)
            sources = set(rk.doc_ids(response.source_nodes))
            rows.append({"provider": provider, "assistant": name, "question": question[:60], "answerable": answerable,
                         "answered": rk.answered(response), "cited": sorted(rk.cited_ids(str(response))),
                         "citations valid": rk.cited_ids(str(response)) <= sources, "s": t.seconds, "answer": str(response)})
    return pd.DataFrame(rows)


results = run_cases(PROVIDER)
results[["assistant", "question", "answerable", "answered", "cited", "citations valid", "s"]]

In [ ]:
# (notebook) Read two answers: one policy answer and one customer answer.
for i in (0, 6):
    print(f"--- {results.loc[i, 'assistant']}: {results.loc[i, 'question']}\n{results.loc[i, 'answer']}\n")

In [ ]:
# (notebook) Check: answerable questions answered with valid citations; unanswerable ones stopped by the floor.
def check_part4():
    make_assistant(POLICY_SEARCH, rk.EU_AFTERSALES, index)  # your TODO must run, not just the reference
    for _, row in results.iterrows():
        assert row["answered"] == row["answerable"], (row["question"], row["answered"])
        assert row["citations valid"], row["question"]
    support = " ".join(results.loc[results["assistant"] == "customer-support", "answer"])
    assert "TSB-" not in support and "After-Sales Technical Support" not in support


check("Part 4 — two assistants from one pattern", check_part4)

## Part 5 — Calibrate the floor and compare providers (20 min)

The floor decides between "answer" and "not covered" **before** any model call. Calibrate it on
labelled questions instead of guessing it.

In [ ]:
# (notebook) The best re-ranked score per question, answerable vs not: where should the floor go?
rerank = rk.reranker(1)
scores = []
for name, (profile, user, questions) in CASES.items():
    retriever = index.as_retriever(similarity_top_k=2 * profile.k, filters=filters_for(user, list(profile.doc_types)))
    for question, answerable in questions:
        best = rerank.postprocess_nodes(retriever.retrieve(question), query_bundle=QueryBundle(question))[0]
        scores.append({"assistant": name, "question": question[:60], "answerable": answerable,
                       "best score": round(float(best.score), 2), "best doc": best.node.metadata["doc_id"]})
scores = pd.DataFrame(scores).sort_values("best score")
lowest_ok = scores.loc[scores["answerable"], "best score"].min()
highest_no = scores.loc[~scores["answerable"], "best score"].max()
print(f"lowest answerable: {lowest_ok} · highest unanswerable: {highest_no} · floor in use: {POLICY_SEARCH.floor}")
scores

In [ ]:
# (notebook) The same assistants on the other provider: retrieval is identical, only the writer changes.
if not HAS_PARTNER:
    print(f"⚠️  {PARTNER} not configured: skipping the comparison")
else:
    both = pd.concat([results, run_cases(PARTNER)])
    print(both.pivot_table(index=["assistant", "question"], columns="provider", values=["answered", "s"], aggfunc="first"))
    print("\n" + both[both["provider"] == PARTNER].iloc[1]["answer"])

📝 **Write-up (3–5 sentences).** Which floor would you ship, and what margin does your calibration
table give? Name one question type that would break it (hint: Lab 02's German query and the
English-only re-ranker). What would you change before connecting the live SharePoint tenant?

> *Your answer here.*

---

**Done.** Answer the checkpoint questions in the lab guide, then go on to
[Lab 04 — Agentic RAG and fine-tuning decisions](../lab-04-agentic-rag-fine-tuning-decisions.md).